# Mobile Money Fraud Detection

**Real-time fraud operations profile for PaySim-style transactions**

Regulatory framework: FFIEC Cybersecurity Examination, FinCEN mobile payment fraud guidance,
CFPB Electronic Fund Transfer Act (Reg E) dispute obligations.

## Part 0 — Setup

In [3]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix,
    precision_recall_fscore_support,
)
warnings.filterwarnings('ignore')

from hugiml import HUGIMLClassifierNative
from hugiml.calibration import evaluate_calibration
from hugiml.metrics import compute_all_metrics
from hugiml.pruning import PatternEditor
from hugiml.governance import generate_model_card

RANDOM_STATE   = 42
DATA_FILE      = 'nb06_mobile_money_fraud_data.csv'
METADATA_FILE  = 'nb06_mobile_money_fraud_metadata.csv'
TARGET         = 'isFraud'
MAX_MODEL_ROWS = 50_000

TYPOLOGY_MAP = {
    'dest_to_amount_ratio':      'Balance Manipulation (dest unchanged / ratio)',
    'balance_change_dest':       'Balance Manipulation (dest unchanged / ratio)',
    'dest_unchanged':            'Balance Manipulation (dest unchanged / ratio)',
    'newbalanceOrig':            'Account Drainage (origin near-zero after txn)',
    'origin_near_zero':          'Account Drainage (origin near-zero after txn)',
    'origin_significant_drop':   'Account Drainage (origin near-zero after txn)',
    'balance_change_orig':       'Account Drainage (origin near-zero after txn)',
    'oldbalanceOrg':             'High-Value Origin Targeting',
    'amount':                    'High-Value Transaction',
    'high_value':                'High-Value Transaction',
    'amount_to_balance_ratio':   'Account Impact Ratio',
    'type':                      'Transaction-Type Risk Profile',
    'is_cash_out':               'Transaction-Type Risk Profile',
    'is_transfer':               'Transaction-Type Risk Profile',
}

THEME = dict(bg='#130c2a',panel='#211642',text='#f5f1ff',muted='#b9aee7',
             accent='#00d7ff',accent2='#ff4fd8',accent3='#ffd166',grid='#3b2a68')
plt.rcParams.update({'figure.facecolor':THEME['bg'],'axes.facecolor':THEME['panel'],
                     'axes.edgecolor':THEME['grid'],'text.color':THEME['text'],'font.size':10})

import hugiml; print('hugiml-core', hugiml.__version__)


hugiml-core 1.1.2


## Part 1 — Data Loading and Quality Review

In [5]:
df = pd.read_csv(DATA_FILE)
metadata = pd.read_csv(METADATA_FILE)
df[TARGET] = df[TARGET].astype(int)

num_feats = df.drop(columns=[TARGET]).select_dtypes(include=[np.number]).columns.tolist()
cat_feats = [c for c in df.drop(columns=[TARGET]).columns if c not in num_feats]
print(f'Full dataset: {len(df):,} rows | {len(num_feats)} numeric + {len(cat_feats)} categorical')
print(f'Fraud count : {int(df[TARGET].sum())} ({df[TARGET].mean():.4%}) — EXTREME IMBALANCE')
print(f'Missing     : {int(df.isna().sum().sum())} | Duplicates: {int(df.duplicated().sum())}')
print('\nTransaction types:')
print(df['type'].value_counts())
print('\nFeature register:')
cols = [c for c in ['column_name','data_type','importance','notes'] if c in metadata.columns]
print(metadata[cols].to_string(index=False))


Full dataset: 100,000 rows | 16 numeric + 1 categorical
Fraud count : 130 (0.1300%) — EXTREME IMBALANCE
Missing     : 0 | Duplicates: 0

Transaction types:
type
CASH_OUT    35194
PAYMENT     33858
CASH_IN     22068
TRANSFER     7909
DEBIT         971
Name: count, dtype: int64

Feature register:
            column_name   data_type            importance                                                                               notes
                   type categorical                  High                                TRANSFER and CASH_OUT show 10-15x higher fraud rates
                 amount   numerical                  High                                           Large transfers more susceptible to fraud
          oldbalanceOrg   numerical              Critical                                       Large balances (>$94K) targeted by fraudsters
         newbalanceOrig   numerical              Critical                   Near-zero balance (<$369) after large transfer indicates dra

## Part 2 — Subsampling and Splits

> **Critical imbalance note.**  Fraud rate is 0.13% — one of the most extreme
> ratios in financial ML.  Key choices:
> - **Primary metric: Average Precision (AP)** — ROC-AUC is misleading at very
>   low base rates because the vast TN pool inflates apparent performance.
> - **50,000 row subsample** (preserving fraud rate) keeps training reproducible.
> - **60/20/20 split** with separate calibration holdout.
> - Isotonic recalibration with only ~13 positives in calibration has high
>   variance; Platt scaling may generalise better in production.


In [7]:
if len(df) > MAX_MODEL_ROWS:
    model_df, _ = train_test_split(df, train_size=MAX_MODEL_ROWS,
                                   stratify=df[TARGET], random_state=RANDOM_STATE)
    model_df = model_df.reset_index(drop=True)
else:
    model_df = df.copy().reset_index(drop=True)

X = model_df.drop(columns=[TARGET])
y = model_df[TARGET]
print(f'Modeling pop: {len(X):,}  |  Fraud: {int(y.sum())} ({y.mean():.4%})')

clf_prep = HUGIMLClassifierNative(B=10, L=1, G=1e-4, topK=100)
X_enc, y_enc = clf_prep.prepareXy(X, y)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_enc, y_enc, test_size=0.40, stratify=y_enc, random_state=RANDOM_STATE
)
X_cal, X_te, y_cal, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE
)
print(f'Train:{len(X_tr):,} (fraud:{y_tr.sum()})  Cal:{len(X_cal):,} (fraud:{y_cal.sum()})  Test:{len(X_te):,} (fraud:{y_te.sum()})')


Modeling pop: 50,000  |  Fraud: 65 (0.1300%)
Train:30,000 (fraud:39)  Cal:10,000 (fraud:13)  Test:10,000 (fraud:13)


## Part 3 — feature_mode Comparison

For real-time fraud blocks, `patterns_only` gives the cleanest Reg E
adverse-action explanation. `original_plus_patterns` may improve AP slightly.


In [9]:
mode_results = {}
for mode in ['patterns_only', 'original_plus_patterns']:
    c = HUGIMLClassifierNative(B=10, L=1, G=1e-4, topK=100, feature_mode=mode)
    Xe, ye = c.prepareXy(X, y)
    Xtr2,Xtmp2,ytr2,ytmp2 = train_test_split(Xe,ye,test_size=0.40,stratify=ye,random_state=RANDOM_STATE)
    Xcal2,Xte2,ycal2,yte2 = train_test_split(Xtmp2,ytmp2,test_size=0.50,stratify=ytmp2,random_state=RANDOM_STATE)
    c.fit(Xtr2, ytr2)
    p2 = c.predict_proba(Xte2)[:,1]
    mode_results[mode] = dict(clf=c,Xtr=Xtr2,Xcal=Xcal2,Xte=Xte2,
                               ytr=ytr2,ycal=ycal2,yte=yte2,proba=p2,
                               auc=roc_auc_score(yte2,p2),ap=average_precision_score(yte2,p2))
    print(f"  {mode:26s}: {len(c.get_hug_features()):3d} patterns | "
          f"AUC={mode_results[mode]['auc']:.4f} | AP={mode_results[mode]['ap']:.4f}")

R = mode_results['patterns_only']
clf,X_tr,X_cal,X_te = R['clf'],R['Xtr'],R['Xcal'],R['Xte']
y_tr,y_cal,y_te,y_score = R['ytr'],R['ycal'],R['yte'],R['proba']
auc,ap = R['auc'],R['ap']

prec_c,rec_c,thr_pr = precision_recall_curve(y_te,y_score)
f1s = np.where((prec_c+rec_c)>0, 2*prec_c*rec_c/(prec_c+rec_c), 0)
f1_idx = int(np.argmax(f1s))
op_thr = float(thr_pr[min(f1_idx,len(thr_pr)-1)])
y_pred = (y_score>=op_thr).astype(int)
tn,fp,fn,tp = confusion_matrix(y_te,y_pred).ravel()
prec_op,rec_op,f1_op,_ = precision_recall_fscore_support(y_te,y_pred,average='binary',zero_division=0)
print(f'\nBaseline @ F1-max threshold {op_thr:.3f}: TP={tp}  FP={fp}  FN={fn}  F1={f1_op:.3f}')


  patterns_only             :  58 patterns | AUC=0.9994 | AP=0.9284
  original_plus_patterns    :  58 patterns | AUC=0.9996 | AP=0.9374

Baseline @ F1-max threshold 0.320: TP=12  FP=1  FN=1  F1=0.923


## Part 4 — Calibration (Pre-Recalibration Baseline)

In [11]:
cal_pre = evaluate_calibration(np.asarray(y_te), y_score)
print(f'ECE={cal_pre.ece:.4f}  MCE={cal_pre.mce:.4f}  Brier={cal_pre.brier_score:.6f}')
print('NOTE: With ~13 test positives, bin-level calibration metrics have high variance.')
print('Brier score is more reliable than ECE at extreme imbalance.')


ECE=0.0011  MCE=0.6798  Brier=0.000357
NOTE: With ~13 test positives, bin-level calibration metrics have high variance.
Brier score is more reliable than ECE at extreme imbalance.


## Part 5 — Interpretability Metrics

In [13]:
interp = compute_all_metrics(clf, X_te)
print(f'n_patterns          : {interp.n_patterns}')
print(f'avg_pattern_length  : {interp.avg_pattern_length:.2f}')
print(f'coverage            : {interp.coverage:.2%}')
print(f'mean_active/sample  : {interp.mean_active_patterns:.2f}')
print(f'explanation_sparsity: {interp.explanation_sparsity:.4f}')
print(f'\nEach flagged transaction can be explained by ~{interp.mean_active_patterns:.0f} active patterns on average.')
print('This satisfies Reg E §1005.11 specific-reason requirements for dispute resolution.')


n_patterns          : 58
avg_pattern_length  : 1.00
coverage            : 100.00%
mean_active/sample  : 6.59
explanation_sparsity: 0.0000

Each flagged transaction can be explained by ~7 active patterns on average.
This satisfies Reg E §1005.11 specific-reason requirements for dispute resolution.


## Part 6 — Fraud Typology Mapping

HUG-IML patterns are labelled with **mobile-money fraud typologies** consistent
with FinCEN guidance on electronic funds transfer fraud:

| Typology | Description |
|----------|-------------|
| **Balance Manipulation** | Destination balance unchanged despite outgoing transfer (dest_to_amount_ratio ≈ 0) |
| **Account Drainage** | Origin account near-zero after transaction (newbalanceOrig ≈ 0) |
| **High-Value Targeting** | Large-balance origin accounts preferentially targeted |
| **Transaction-Type Profile** | CASH_OUT / TRANSFER at elevated risk vs PAYMENT / CASH_IN |


In [15]:
importances = clf.feature_importances().copy()

def _typology(pat):
    for kw,t in TYPOLOGY_MAP.items():
        if kw in pat: return t
    return 'Other'

importances['typology']  = importances['pattern'].apply(_typology)
importances['direction'] = np.where(importances['coefficient']>=0,'fraud-signal','legitimacy-signal')

top15 = importances.nlargest(15,'abs_coefficient')
print('Top 15 patterns with typology labels:')
print('='*95)
for _,row in top15.iterrows():
    arrow = '▲' if row['coefficient']>0 else '▼'
    print(f"  {arrow} {row['pattern']:44s} {row['coefficient']:+.4f}  sup={row['support']:.1%}  [{row['typology']}]")
print('='*95)
print('\nPatterns per typology:')
print(importances.groupby('typology').size().sort_values(ascending=False))


Top 15 patterns with typology labels:
  ▼ dest_to_amount_ratio=[0.9998,1)              -3.2365  sup=10.0%  [Balance Manipulation (dest unchanged / ratio)]
  ▼ balance_change_dest=[-4.612e+05,-677.4)      -2.5788  sup=10.0%  [Balance Manipulation (dest unchanged / ratio)]
  ▼ type=PAYMENT                                 -2.0910  sup=33.9%  [Transaction-Type Risk Profile]
  ▼ balance_change_dest=[-677.4,-167.1)          -2.0539  sup=10.0%  [Balance Manipulation (dest unchanged / ratio)]
  ▲ oldbalanceOrg=[9.403e+04,5e+07)              +1.8113  sup=10.0%  [High-Value Origin Targeting]
  ▼ balance_change_dest=[-167.1,-30.8)           -1.7749  sup=10.0%  [Balance Manipulation (dest unchanged / ratio)]
  ▲ newbalanceOrig=[0,363.7)                     +1.5822  sup=30.0%  [Account Drainage (origin near-zero after txn)]
  ▲ dest_to_amount_ratio=[-0.2513,0.9178)        +1.4240  sup=10.0%  [Balance Manipulation (dest unchanged / ratio)]
  ▲ oldbalanceOrg=[2.824e+04,9.403e+04)          +1.4194  su

## Part 7 — Pattern Review and Pruning

**Reg E / FFIEC context:** Each pattern that fires on a blocked / flagged transaction
becomes a specific adverse-action reason. Patterns must be defensible to customers
and examiners:

- **RETAIN** balance-manipulation and account-drainage patterns (clear fraud evidence)
- **RETAIN** transaction-type legitimacy signals (PAYMENT, CASH_IN are strong indicators)
- **REMOVE** low-support patterns (<8%) that are unstable across payment windows
- No protected demographic attributes exist in this dataset


In [17]:
editor = PatternEditor(clf, operator_name='fraud-ops-review')
pats_df = editor.list_patterns()
print(f'Patterns before review: {len(pats_df)}')

editor.remove_low_support(
    min_support=0.08,
    reason='Low-support patterns are unstable across payment windows; '
           'remove to improve model reliability in production'
)
print(f'Patterns after low-support removal: {len(editor.list_patterns())}')

editor.refit(X_tr, y_tr)
editor.calibrate(X_cal, y_cal, method='isotonic')
clf_pruned = editor.finalize()

proba_pruned = clf_pruned.predict_proba(X_te)[:,1]
auc_pruned   = roc_auc_score(y_te, proba_pruned)
ap_pruned    = average_precision_score(y_te, proba_pruned)
cal_post     = evaluate_calibration(np.asarray(y_te), proba_pruned)

prec_pp,rec_pp,thr_pp2 = precision_recall_curve(y_te,proba_pruned)
f1pp = np.where((prec_pp+rec_pp)>0,2*prec_pp*rec_pp/(prec_pp+rec_pp),0)
f1p_idx = int(np.argmax(f1pp))
op_thr_p = float(thr_pp2[min(f1p_idx,len(thr_pp2)-1)])
y_pred_p = (proba_pruned>=op_thr_p).astype(int)
tn_p,fp_p,fn_p,tp_p = confusion_matrix(y_te,y_pred_p).ravel()
prec_p,rec_p,f1_p,_ = precision_recall_fscore_support(y_te,y_pred_p,average='binary',zero_division=0)

print(f'\nAfter pruning + isotonic recalibration:')
print(f'  Patterns : {len(clf_pruned.get_hug_features())} (was {len(clf.get_hug_features())})')
print(f'  AUC-ROC  : {auc_pruned:.4f}  (was {auc:.4f})')
print(f'  Avg Prec : {ap_pruned:.4f}  (was {ap:.4f})')
print(f'  ECE      : {cal_post.ece:.4f}  (was {cal_pre.ece:.4f})')
print(f'  F1 @ op  : {f1_p:.3f}  precision={prec_p:.2%}  recall={rec_p:.2%}')

audit_js = json.loads(editor.audit_report())
print(f'\nAudit: removed={audit_js["diff"]["n_removed"]}, calibrated={audit_js["calibration"]["applied"]}')
for r in audit_js.get('removals',[]):
    print(f'  [{r["reason"][:80]}]  → {r["pattern_labels"]}')


Patterns before review: 58
Patterns after low-support removal: 57

After pruning + isotonic recalibration:
  Patterns : 57 (was 58)
  AUC-ROC  : 0.9996  (was 0.9994)
  Avg Prec : 0.8948  (was 0.9284)
  ECE      : 0.0008  (was 0.0011)
  F1 @ op  : 0.917  precision=100.00%  recall=84.62%

Audit: removed=1, calibrated=True
  [Low-support patterns are unstable across payment windows; remove to improve mode]  → ['type=TRANSFER']


## Part 8 — Threshold Analysis (Reg E Framing)

Reg E §1005.6(b) limits consumer liability but obligates institutions to investigate
disputed errors within 10 days. Threshold selection must balance customer friction
(false positives = blocked legitimate transactions) against fraud loss exposure.


In [19]:
thresh_rows = []
for thr in np.linspace(0.01,0.99,49):
    pred = (proba_pruned>=thr).astype(int)
    if pred.sum()==0: continue
    tn_t,fp_t,fn_t,tp_t = confusion_matrix(np.asarray(y_te),pred,labels=[0,1]).ravel()
    prec_t,rec_t,f1_t,_ = precision_recall_fscore_support(y_te,pred,average='binary',zero_division=0)
    thresh_rows.append({'threshold':thr,'n_flagged':int(pred.sum()),
                        'tp':tp_t,'fp':fp_t,'fn':fn_t,
                        'precision':prec_t,'recall':rec_t,'f1':f1_t,
                        'flag_rate':pred.mean()})
threshold_table = pd.DataFrame(thresh_rows)

print('Operating points:')
for target_rec in [0.70, 0.80, 0.90]:
    rows_ge = threshold_table[threshold_table['recall']>=target_rec]
    if len(rows_ge):
        row = rows_ge.iloc[-1]
        print(f"  {int(target_rec*100)}% recall → thr={row.threshold:.3f}  "
              f"precision={row.precision:.2%}  flags={int(row.n_flagged)}  FP/TP={row.fp/max(row.tp,1):.2f}")

threshold_table.round(4)


Operating points:
  70% recall → thr=0.990  precision=100.00%  flags=11  FP/TP=0.00
  80% recall → thr=0.990  precision=100.00%  flags=11  FP/TP=0.00
  90% recall → thr=0.500  precision=48.00%  flags=25  FP/TP=1.08


,threshold,n_flagged,tp,fp,fn,precision,recall,f1,flag_rate
0,0.0100,85,13,72,0,0.1529,1.0000,0.2653,0.0085
1,0.0304,27,12,15,1,0.4444,0.9231,0.6000,0.0027
2,0.0508,27,12,15,1,0.4444,0.9231,0.6000,0.0027
3,0.0712,27,12,15,1,0.4444,0.9231,0.6000,0.0027
4,0.0917,27,12,15,1,0.4444,0.9231,0.6000,0.0027
5,0.1121,27,12,15,1,0.4444,0.9231,0.6000,0.0027
6,0.1325,27,12,15,1,0.4444,0.9231,0.6000,0.0027
7,0.1529,27,12,15,1,0.4444,0.9231,0.6000,0.0027
8,0.1733,27,12,15,1,0.4444,0.9231,0.6000,0.0027
9,0.1938,27,12,15,1,0.4444,0.9231,0.6000,0.0027


## Part 9 — Subgroup Flag-Rate Audit

In [21]:
GROUP_COLS = ['type','is_cash_out','is_transfer','dest_unchanged','origin_near_zero','high_value']
test_idx = X_te.index if hasattr(X_te,'index') else pd.Index(range(len(y_te)))
raw_test = X.loc[test_idx].copy()
af = raw_test.copy()
af['_actual'] = np.asarray(y_te).astype(int)
af['_score']  = proba_pruned
af['_flag']   = y_pred_p

sub_rows=[]
for col in GROUP_COLS:
    if col not in af.columns: continue
    for level in af[col].value_counts().index[:10]:
        g = af[af[col].eq(level)]
        if len(g)<20: continue
        yg=g['_actual'].to_numpy(); fg=g['_flag'].to_numpy()
        auc_g = roc_auc_score(yg,g['_score']) if len(np.unique(yg))==2 else np.nan
        tn_g,fp_g,fn_g,tp_g = confusion_matrix(yg,fg,labels=[0,1]).ravel()
        sub_rows.append({'feature':col,'segment':str(level),'n':len(g),
                         'base_rate':yg.mean(),'flag_rate':fg.mean(),'auc':auc_g,
                         'precision':tp_g/max(tp_g+fp_g,1),'recall':tp_g/max(tp_g+fn_g,1),
                         'fpr':fp_g/max(fp_g+tn_g,1)})

subgroup_audit = pd.DataFrame(sub_rows)
subgroup_audit.sort_values(['feature','n'],ascending=[True,False]).round(4)


,feature,segment,n,base_rate,flag_rate,auc,precision,recall,fpr
9,dest_unchanged,0,7031,0.0003,0.0000,0.9990,0.0,0.0000,0.0
10,dest_unchanged,1,2969,0.0037,0.0037,1.0000,1.0,1.0000,0.0
13,high_value,0,9929,0.0008,0.0006,0.9994,1.0,0.7500,0.0
14,high_value,1,71,0.0704,0.0704,1.0000,1.0,1.0000,0.0
5,is_cash_out,0,6505,0.0009,0.0008,0.9999,1.0,0.8333,0.0
6,is_cash_out,1,3495,0.0020,0.0017,0.9997,1.0,0.8571,0.0
7,is_transfer,0,9222,0.0008,0.0007,0.9999,1.0,0.8571,0.0
8,is_transfer,1,778,0.0077,0.0064,1.0000,1.0,0.8333,0.0
11,origin_near_zero,0,6766,0.0000,0.0000,NaN,0.0,0.0000,0.0
12,origin_near_zero,1,3234,0.0040,0.0034,0.9993,1.0,0.8462,0.0


## Part 10 — Score Decile Lift

In [23]:
score_df = pd.DataFrame({'actual':np.asarray(y_te).astype(int),'score':proba_pruned})
score_df['decile'] = pd.qcut(score_df['score'].rank(method='first'),10,
                               labels=range(1,11)).astype(int)
decile = score_df.groupby('decile').agg(
    n=('actual','size'),event_rate=('actual','mean'),
    avg_score=('score','mean'),events=('actual','sum')
).reset_index()
decile['capture_pct'] = decile['events']/max(decile['events'].sum(),1)
decile = decile.sort_values('decile',ascending=False)

base = float(score_df['actual'].mean())
top_d = float(decile[decile['decile'].eq(10)]['event_rate'].iloc[0])
print(f'Top decile: {top_d:.2%}  ({top_d/max(base,1e-9):.0f}× base rate of {base:.4%})')
decile.round(4)


Top decile: 1.30%  (10× base rate of 0.1300%)


,decile,n,event_rate,avg_score,events,capture_pct
9,10,1000,0.013,0.021,13,1.0
8,9,1000,0.000,0.000,0,0.0
7,8,1000,0.000,0.000,0,0.0
6,7,1000,0.000,0.000,0,0.0
5,6,1000,0.000,0.000,0,0.0
4,5,1000,0.000,0.000,0,0.0
3,4,1000,0.000,0.000,0,0.0
2,3,1000,0.000,0.000,0,0.0
1,2,1000,0.000,0.000,0,0.0
0,1,1000,0.000,0.000,0,0.0


## Part 11 — Covariate Drift Detection (PSI)

In [25]:
def compute_psi(expected,actual,buckets=10):
    rows=[]
    common=expected.select_dtypes(include=[np.number]).columns.intersection(actual.columns)
    for col in common:
        edges=np.percentile(expected[col].dropna(),np.linspace(0,100,buckets+1))
        edges[0]=-np.inf; edges[-1]=np.inf
        ep=np.maximum(np.histogram(expected[col],bins=edges)[0]/len(expected),1e-6)
        ap_=np.maximum(np.histogram(actual[col],bins=edges)[0]/len(actual),1e-6)
        psi=float(np.sum((ap_-ep)*np.log(ap_/ep)))
        rows.append({'feature':col,'psi':round(psi,4),
                     'status':'STABLE' if psi<0.10 else 'WARNING' if psi<0.25 else 'SHIFT'})
    return pd.DataFrame(rows).sort_values('psi',ascending=False)

rng_d = np.random.default_rng(55)
n_d   = 2000
X_raw_train = X[num_feats].iloc[:30000]
X_raw_drift = pd.DataFrame({
    'amount':                  rng_d.lognormal(12.5,1.5,n_d),
    'oldbalanceOrg':           rng_d.lognormal(12.0,1.2,n_d),
    'oldbalanceDest':          rng_d.uniform(0,100,n_d),
    'newbalanceOrig':          rng_d.uniform(0,200,n_d),
    'newbalanceDest':          rng_d.uniform(0,100,n_d),
    'balance_change_orig':     -rng_d.lognormal(12.0,1.2,n_d),
    'balance_change_dest':     rng_d.uniform(-50,50,n_d),
    'amount_to_balance_ratio': rng_d.uniform(0.8,2.0,n_d),
    'dest_to_amount_ratio':    rng_d.uniform(-0.1,0.1,n_d),
})
X_raw_drift = X_raw_drift[[c for c in X_raw_drift.columns if c in X_raw_train.columns]]

psi_df = compute_psi(X_raw_train, X_raw_drift)
print('PSI vs simulated fraud-wave / account-drainage shift window:')
print(psi_df.to_string(index=False))


PSI vs simulated fraud-wave / account-drainage shift window:
                feature     psi status
   dest_to_amount_ratio 12.4339  SHIFT
    balance_change_orig 12.4339  SHIFT
         oldbalanceDest 11.0837  SHIFT
amount_to_balance_ratio 11.0738  SHIFT
                 amount 10.7387  SHIFT
         newbalanceDest 10.4978  SHIFT
    balance_change_dest  9.0225  SHIFT
         newbalanceOrig  8.9017  SHIFT
          oldbalanceOrg  7.5161  SHIFT


## Part 12 — Model Governance and Artifact Export

In [27]:
card = generate_model_card(
    clf_pruned,
    model_id='mobile-money-fraud-v1.0',
    intended_use=(
        'Real-time transaction scoring for mobile-money fraud detection. '
        'Output feeds real-time block/flag decisions and Reg E dispute triage.'
    ),
    out_of_scope_use=(
        'Not validated for card-present fraud. '
        'Not for account-takeover detection without authentication features.'
    ),
    training_data_description=(
        f'PaySim-style synthetic dataset, {len(model_df):,} transactions, '
        f'{int(y.sum())} fraud ({y.mean():.4%}). 60/20/20 split.'
    ),
    evaluation_data_description=f'Stratified 20% holdout; {int(y_te.sum())} fraud transactions.',
    performance_metrics={
        'AUC-ROC':        round(auc_pruned,4),
        'AvgPrecision':   round(ap_pruned,4),
        'ECE':            round(cal_post.ece,4),
        'F1_at_op':       round(float(f1_p),4),
        'Precision_at_op':round(float(prec_p),4),
        'Recall_at_op':   round(float(rec_p),4),
    },
    limitations=[
        'Trained on synthetic data — production requires validation on real labels.',
        'Extreme imbalance (0.13%) means calibration estimates have high variance.',
        'Model does not capture account-takeover pre-cursors.',
        'Monthly PSI monitoring required — fraud patterns evolve rapidly.',
    ],
    ethical_considerations=(
        'No protected demographic attributes in this dataset. '
        'Reg E §1005.11 requires specific adverse-action reasons — '
        'pattern-level explanations satisfy this. '
        'Dispute investigation process must remain human-led per Reg E.'
    ),
)

print(card.to_markdown())
card.save('nb06_mobile_money_fraud_model_card.json')
editor.save_audit_report('nb06_mobile_money_fraud_audit_trail.json')
importances.to_csv('nb06_mobile_money_fraud_pattern_inventory.csv',index=False)
threshold_table.to_csv('nb06_mobile_money_fraud_threshold_grid.csv',index=False)
subgroup_audit.to_csv('nb06_mobile_money_fraud_subgroup_audit.csv',index=False)
psi_df.to_csv('nb06_mobile_money_fraud_psi_report.csv',index=False)
print('\n✓ Governance artifacts saved.')


# Model Card: mobile-money-fraud-v1.0

**Type:** HUGIMLClassifierNative  
**License:** Apache-2.0  
**Created:** 2026-05-29T01:23:13Z  
**Framework:** hugiml-core 1.1.2

## Reference

Krishnamoorthy, S. (2024). Interpretable Classifier Models for Decision Support Using High Utility Gain Patterns. IEEE Access, 12, 126088-126107. DOI: 10.1109/ACCESS.2024.3455563

## Intended Use

Real-time transaction scoring for mobile-money fraud detection. Output feeds real-time block/flag decisions and Reg E dispute triage.

## Out-of-Scope Use

Not validated for card-present fraud. Not for account-takeover detection without authentication features.

## Training Data

PaySim-style synthetic dataset, 50,000 transactions, 65 fraud (0.1300%). 60/20/20 split.

## Evaluation Data

Stratified 20% holdout; 13 fraud transactions.

## Hyperparameters

- **B**: 10
- **L**: 1
- **G**: 0.0001
- **topK**: 100
- **adaptive_binning**: False
- **feature_mode**: patterns_only

## Performance Metrics

- **AUC-ROC**: 0